In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.jfk_flights_final_data AS
WITH flights_parsed AS (
  SELECT
    *,
    COALESCE(
      try_to_timestamp(CAST(FL_DATE AS STRING), 'M/d/yyyy h:mm:ss a'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd HH:mm:ss'),
      try_to_timestamp(CAST(FL_DATE AS STRING), 'yyyy-MM-dd')
    ) AS fl_date_ts,
    CAST(CRS_DEP_TIME AS INT) AS crs_dep_int
  FROM flight_delay_predictor_project.default.jfk_flight_data
),
flights_hourly AS (
  SELECT
    *,
    timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts) AS dep_hour,
    timestampadd(MINUTE, crs_dep_int % 100,
                 timestampadd(HOUR, crs_dep_int DIV 100, fl_date_ts)) AS crs_dep_ts
  FROM flights_parsed
)
SELECT
  f.*,
  -- Weather at departure hour
  w0.* EXCEPT (time),
  -- Weather 2 hour after
  w2.temperature_2m        AS temperature_2m_plus2h,
  w2.precipitation         AS precipitation_plus2h,
  w2.snowfall              AS snowfall_plus2h,
  w2.wind_speed_10m        AS wind_speed_10m_plus2h,
  w2.wind_gusts_10m        AS wind_gusts_10m_plus2h,
  w2.cloud_cover           AS cloud_cover_plus2h,
  w2.weather_code          AS weather_code_plus2h,
  -- Weather 5 hours after
  w5.temperature_2m        AS temperature_2m_plus5h,
  w5.precipitation         AS precipitation_plus5h,
  w5.snowfall              AS snowfall_plus5h,
  w5.wind_speed_10m        AS wind_speed_10m_plus5h,
  w5.wind_gusts_10m        AS wind_gusts_10m_plus5h,
  w5.cloud_cover           AS cloud_cover_plus5h,
  w5.weather_code          AS weather_code_plus5h,
    -- Weather 7 hours after
  w7.temperature_2m        AS temperature_2m_plus7h,
  w7.precipitation         AS precipitation_plus7h,
  w7.snowfall              AS snowfall_plus7h,
  w7.wind_speed_10m        AS wind_speed_10m_plus7h,
  w7.wind_gusts_10m        AS wind_gusts_10m_plus7h,
  w7.cloud_cover           AS cloud_cover_plus7h,
  w7.weather_code          AS weather_code_plus7h
FROM flights_hourly f
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w0
  ON f.dep_hour = w0.time
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w2
  ON timestampadd(HOUR, 2, f.dep_hour) = w2.time
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w5
  ON timestampadd(HOUR, 5, f.dep_hour) = w5.time
LEFT JOIN flight_delay_predictor_project.default.jfk_weather_history w7
  ON timestampadd(HOUR, 7, f.dep_hour) = w7.time;

In [0]:
%sql
SELECT *
FROM flight_delay_predictor_project.default.jfk_flights_final_data

In [0]:
%sql
SELECT weather_delay, temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m, wind_gusts_10m, cloud_cover, pressure_msl, weather_code, temperature_2m_plus2h, precipitation_plus2h, snowfall_plus2h, wind_speed_10m_plus2h, wind_gusts_10m_plus2h, cloud_cover_plus2h, weather_code_plus2h, temperature_2m_plus5h, precipitation_plus5h, snowfall_plus5h, wind_speed_10m_plus5h, wind_gusts_10m_plus5h, cloud_cover_plus5h, weather_code_plus5h, temperature_2m_plus7h, precipitation_plus7h, snowfall_plus7h, wind_speed_10m_plus7h, wind_gusts_10m_plus7h,
cloud_cover_plus7h, weather_code_plus7h
FROM flight_delay_predictor_project.default.jfk_flights_final_data
WHERE fl_date_ts < '2026-07-30';

In [0]:
%sql
CREATE OR REPLACE TABLE flight_delay_predictor_project.default.Final_Prediction_Table 
SELECT weather_delay, temperature_2m, relative_humidity_2m, precipitation, snowfall, wind_speed_10m, wind_direction_10m, wind_gusts_10m, cloud_cover, pressure_msl, weather_code, temperature_2m_plus2h, precipitation_plus2h, snowfall_plus2h, wind_speed_10m_plus2h, wind_gusts_10m_plus2h, cloud_cover_plus2h, weather_code_plus2h, temperature_2m_plus5h, precipitation_plus5h, snowfall_plus5h, wind_speed_10m_plus5h, wind_gusts_10m_plus5h, cloud_cover_plus5h, weather_code_plus5h, temperature_2m_plus7h, precipitation_plus7h, snowfall_plus7h, wind_speed_10m_plus7h, wind_gusts_10m_plus7h,
cloud_cover_plus7h, weather_code_plus7h
FROM flight_delay_predictor_project.default.jfk_flights_final_data
WHERE fl_date_ts < '2026-07-30';

In [0]:
%pip install seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.utils import resample

In [0]:
df = spark.table('flight_delay_predictor_project.default.final_prediction_table').toPandas()

In [0]:
from sklearn.model_selection import train_test_split
df["weather_delay"] = df["weather_delay"].fillna(0)

In [0]:
df = spark.createDataFrame(df)

In [0]:
from pyspark.sql import Window, functions as F

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select("temperature_2m", "relative_humidity_2m", "precipitation", "snowfall", "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m", "cloud_cover", "pressure_msl", "weather_code", "delay_in_hr",
"temperature_2m_plus2h", "precipitation_plus2h", "snowfall_plus2h", "wind_speed_10m_plus2h", "wind_gusts_10m_plus2h", "cloud_cover_plus2h", "weather_code_plus2h",
"temperature_2m_plus5h", "precipitation_plus5h", "snowfall_plus5h", "wind_speed_10m_plus5h", "wind_gusts_10m_plus5h", "cloud_cover_plus5h", "weather_code_plus5h",
"temperature_2m_plus7h", "precipitation_plus7h", "snowfall_plus7h", "wind_speed_10m_plus7h", "wind_gusts_10m_plus7h", "cloud_cover_plus7h", "weather_code_plus7h"))

In [0]:

hr_delay = (df.withColumn("delay_in_hr", 
F.when(F.col("weather_delay") < 1, "no delay")
.when(F.col("weather_delay") < 60, "less than 1") 
.when(F.col("weather_delay") < 120, "1+")
.when(F.col("weather_delay") < 180, "2+")
.when(F.col("weather_delay") < 240, "3+")
.when(F.col("weather_delay") < 300, "4+")
.when(F.col("weather_delay") < 360, "5+")
.when(F.col("weather_delay") < 420, "6+")
.when(F.col("weather_delay") < 480, "7+")
.when(F.col("weather_delay") < 540, "8+")
.when(F.col("weather_delay") < 600, "9+")
.otherwise("10+")) 
.select( "snowfall", "wind_speed_10m", "wind_gusts_10m", "cloud_cover", "weather_code", "delay_in_hr", "precipitation_plus2h", "snowfall_plus2h", "wind_speed_10m_plus2h", "wind_gusts_10m_plus2h", "cloud_cover_plus2h", "weather_code_plus2h", "precipitation_plus5h", "snowfall_plus5h", "wind_speed_10m_plus5h", "wind_gusts_10m_plus5h", "cloud_cover_plus5h", "weather_code_plus5h", "precipitation_plus7h", "snowfall_plus7h", "wind_speed_10m_plus7h", "wind_gusts_10m_plus7h", "cloud_cover_plus7h", "weather_code_plus7h"))

In [0]:
hr_delay = hr_delay.toPandas()
print(hr_delay)

In [0]:
flights_1 = hr_delay[hr_delay['delay_in_hr'] != 'no delay']
flights_2 = hr_delay[hr_delay['delay_in_hr'] == 'no delay']
print(flights_1.shape)
print(flights_2.shape)

In [0]:
flights_1_upsampled = resample(flights_1,random_state=42,n_samples=101915,replace=True)
flights_upsampled = pd.concat([flights_1_upsampled,flights_2])

In [0]:
from sklearn.tree import DecisionTreeClassifier
X = flights_upsampled.drop('delay_in_hr',axis=1)
y = flights_upsampled['delay_in_hr']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20,random_state=42)
dtree = DecisionTreeClassifier()

In [0]:
dtree.fit(X_train,y_train)

In [0]:
from sklearn.metrics import classification_report, confusion_matrix
predictions = dtree.predict(X_test)
print(classification_report(y_test,predictions))
print(confusion_matrix(y_test,predictions))

In [0]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, predictions)


In [0]:
classifier = DecisionTreeClassifier()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy_score(y_test, y_pred)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No delay", "Delay"],
            yticklabels=["No delay", "Delay"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix")
plt.show()

In [0]:
import pandas as pd
from sklearn.metrics import classification_report

report = classification_report(y_test, predictions,
                               output_dict=True)

report_df = pd.DataFrame(report).T.drop(index="accuracy")       # accuracy row is a single number, so drop it
report_df = report_df[["precision", "recall", "f1-score"]]      # leave out support (it's a count, not 0 to 1)

sns.heatmap(report_df, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Classification report")
plt.show()